# **2. Classification of natural disaster Tweets: Cleaning**

The dataset used in this project comes from the Disaster Tweets dataset, which contains a collection of tweets labeled according to whether they refer to a real disaster event or not. The goal of the dataset is to support the development of machine learning models capable of automatically identifying disaster-related information from social media posts.

In natural language processing tasks, raw text data often contains noise such as punctuation, links, mentions, and other elements that may not contribute meaningful information for machine learning models. Therefore, applying different **text preprocessing strategies** can help improve model performance by standardizing and simplifying the textual input.

In this project, three different preprocessing configurations were implemented and compared to evaluate their impact on the classification performance.



## 0. Imports

In [40]:
import re
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os

In [41]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ale\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [42]:
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Ale\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Ale\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## 1. Load dataset

In [60]:
df = pd.read_csv('train.csv')


## 2. Cleaning Data

### **Duplicates**

In [61]:
def delete_duplicates(df):

    duplicates = df.duplicated(subset='text').sum()
    retweets = df['text'].str.startswith('RT')
    retweets_count = retweets.sum()

    if duplicates != 0 or retweets_count != 0:
        print(f"There are {duplicates} duplicate tweets")
        print(f"There are {retweets_count} retweets")
    else:
        print("The dataset does not have duplicate tweets or retweets")

    # Remove duplicates
    df = df.drop_duplicates(subset='text')

    # Remove retweets
    df = df[~df['text'].str.startswith('RT')]

    return df

In [62]:
df = delete_duplicates(df)

There are 110 duplicate tweets
There are 46 retweets


### **Null values**

In [63]:
df.isnull().sum()

id             0
keyword       55
location    2463
text           0
target         0
dtype: int64

In [64]:
df = df.drop(['keyword', 'location', 'id'], axis=1)

### **HTML format**

In [65]:
from html.parser import HTMLParser

class HTMLStripper(HTMLParser):
    def __init__(self):
        self.reset()
        self.strict = False
        self.convert_charrefs = True
        self.fed = []

    def handle_data(self, d):
        self.fed.append(d)

    def get_data(self):
        return ''.join(self.fed)

def remove_html(text):
    s = HTMLStripper()
    s.feed(text)
    return s.get_data()

In [66]:
def remove_emoji(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

In [67]:
df['text'] = df['text'].apply(remove_html)
df['text'] = df['text'].apply(remove_emoji)

## **Basic Text Cleaning**

The first configuration applies a set of basic preprocessing steps aimed at standardizing the text and removing unnecessary elements. The following operations are performed:

* Conversion of all text to **lowercase** in order to avoid treating the same word with different capitalization as different tokens.
* **Removal of URLs**, since links generally do not provide useful semantic information for the classification task.
* **Removal of mentions** (e.g., `@username`), which typically refer to users rather than meaningful textual content.
* **Removal of punctuation marks**, such as commas, periods, and special characters, to simplify the text representation.

This configuration establishes a baseline preprocessing pipeline.



In [68]:
lematizer = WordNetLemmatizer()

In [69]:
def clean_tweets(texts):
    cleaned = []
    for tweet in texts:

        tweet = tweet.lower()
        tweet = re.sub(r"http\S+", "", tweet)
        tweet = re.sub(r"@\w+", "", tweet)

        tweet = re.sub(r"#", "", tweet)
        cleaned.append(tweet)

    return cleaned

def stopwords_clean(tweets):
    stop_words = set(stopwords.words('english'))
    cleaned = []

    for tweet in tweets:
        filtered_words = []
        for word in tweet.split():
            if word not in stop_words:
                filtered_words.append(word)
        cleaned.append(" ".join(filtered_words))
    return pd.Series(cleaned)

def lemmatization(tweets):
    lemmatizer = WordNetLemmatizer()
    cleaned = []

    for tweet in tweets:
        lemmatized_words = []
        for word in tweet.split():
            lemmatized_words.append(lemmatizer.lemmatize(word))
        cleaned.append(" ".join(lemmatized_words))

    return pd.Series(cleaned)


def preprocess_text(tweets):
    tweets = cleaning(tweets)
    tweets = stopwords_clean(tweets)
    #tweets = lemmatization(tweets)

    return tweets



In [70]:
df['text'] = preprocess_text(df['text'])

In [71]:
df = df.dropna()

In [72]:
def save_cleaned_data(df, filename):
    path = 'Cleaned_Data'
    if not os.path.exists(path):
        os.makedirs(path)
    df.to_csv(os.path.join(path, filename), index=False)
    print(f"Cleaned data saved to '{filename}' in the '{path}' directory.")

In [73]:
save_cleaned_data(df, 'cleaned_data_conf.csv')


Cleaned data saved to 'cleaned_data_conf.csv' in the 'Cleaned_Data' directory.
